# 10. 제조회사(`mnfctur_cmpny_nm`)별 리스테리아 검출

- 원자료: `listeria_body.parquet` (676,537행)
- 업체명은 자유 입력 문자열이고 결측이 많다 → 결측 현황부터 보고, 표기 차이를 정규화한 이름(`cmp_key`)으로도 묶어 본다
- 버섯 행 = `prduct_nm` 에 버섯·MUSHROOM·Flammulina·ENOKI·팽이 포함 (노트북 9 와 같은 기준)

In [1]:
import re
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 60)

df = pd.read_parquet(r"C:\cheon\cheon_wokespace\listeria2026\리스테리아ML\data\listeria_body.parquet")
df["pos"] = (df["judge_result"] == "검출").astype(int)
df["is_mush"] = df["prduct_nm"].astype(str).str.contains("버섯|MUSHROOM|Flammulina|ENOKI|팽이", case=False, regex=True)
print(f"전체 {len(df):,}행 / 검출 {df['pos'].sum():,}건 / 검출률 {df['pos'].mean() * 100:.3f}%")

전체 676,537행 / 검출 1,969건 / 검출률 0.291%


## 10-1. 업체명 결측 현황 — 결측 행과 비결측 행의 검출률이 다른가

In [2]:
raw = df["mnfctur_cmpny_nm"].astype(object)
df["cmp_raw"] = raw.where(raw.notna(), None)
df["cmp_raw"] = df["cmp_raw"].map(lambda s: s.strip() if isinstance(s, str) else s)
df.loc[df["cmp_raw"].isin(["", "-", "nan", "None", "없음", "미상"]), "cmp_raw"] = None   # 빈값 성격의 문자열도 결측 취급
df["cmp_missing"] = df["cmp_raw"].isna()

miss = df.groupby("cmp_missing").agg(행수=("pos", "size"), 검출=("pos", "sum"), 버섯행=("is_mush", "sum"))
miss["행비중(%)"] = (miss["행수"] / miss["행수"].sum() * 100).round(2)
miss["검출비중(%)"] = (miss["검출"] / miss["검출"].sum() * 100).round(2)
miss["검출률(%)"] = (miss["검출"] / miss["행수"] * 100).round(3)
miss.index = miss.index.map({False: "업체명 있음", True: "업체명 결측"})
display(miss)

print("[결측 여부 × 연도] 행수")
display(pd.crosstab(df["mnfctur_de"].dt.year, df["cmp_missing"].map({False: "있음", True: "결측"}), margins=True))
print("[결측 여부 × src_route / data_source] 행수")
display(pd.crosstab([df["src_route"].astype(str), df["data_source"].astype(str)], df["cmp_missing"].map({False: "있음", True: "결측"})))

,행수,검출,버섯행,행비중(%),검출비중(%),검출률(%)
cmp_missing,,,,,,
업체명 있음,344127,535,4372,50.87,27.17,0.155
업체명 결측,332410,1434,5479,49.13,72.83,0.431


[결측 여부 × 연도] 행수


cmp_missing,결측,있음,All
mnfctur_de,,,
2015,11,378,389
2016,2537,7691,10228
2017,29436,14438,43874
2018,32846,20868,53714
2019,34609,26141,60750
2020,34885,31129,66014
2021,34159,35422,69581
2022,34407,39002,73409
2023,33076,43956,77032


[결측 여부 × src_route / data_source] 행수


,cmp_missing,결측,있음
src_route,data_source,,
domestic,식약처,298447,330448
overseas,식약처,33963,13679


## 10-2. 업체명 정규화

같은 업체가 `농업회사법인(주)머쉬앤파머스` / `농업회사법인 주식회사 머쉬앤파머스` 처럼 다르게 적혀 있다.
아래 규칙으로 `cmp_key` 를 만든다 (원본 `mnfctur_cmpny_nm` 은 바꾸지 않음).
- 법인 형태 표기 제거: 주식회사, (주), ㈜, 유한회사, (유), 농업회사법인, 영농조합법인, 합자회사, 합명회사, 사단법인, 재단법인, CO., LTD, INC, CORP 등
- 괄호 안 내용 제거, 공백·구두점 제거, 영문 대문자화

In [3]:
CORP_PAT = re.compile(
    r"농업회사법인|영농조합법인|어업회사법인|주식회사|유한회사|합자회사|합명회사|사단법인|재단법인|"
    r"\(주\)|\(유\)|㈜|\(株\)|CO\.?,?\s*LTD\.?|CO\.|LTD\.?|INC\.?|CORP\.?|CORPORATION|LIMITED",
    flags=re.I)


def norm_cmp(s):
    if not isinstance(s, str):
        return None
    t = CORP_PAT.sub(" ", s)
    t = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", t)      # 괄호 내용(지점명 등) 제거
    t = re.sub(r"[\s\.,·\-_/&'\"]+", "", t).upper()
    return t or None


df["cmp_key"] = df["cmp_raw"].map(norm_cmp)
n_raw, n_key = df["cmp_raw"].nunique(), df["cmp_key"].nunique()
print(f"고유 업체명: 원본 {n_raw:,}개 → 정규화 {n_key:,}개 ({n_raw - n_key:,}개 병합)")

# 병합이 많이 일어난 키 예시 (원본 표기가 여러 개인 키)
variants = (df.dropna(subset=["cmp_key"]).groupby("cmp_key")["cmp_raw"]
              .agg(표기수="nunique", 표기예=lambda s: " | ".join(s.value_counts().index[:4])))
display(variants.sort_values("표기수", ascending=False).head(20))

고유 업체명: 원본 19,225개 → 정규화 16,847개 (2,378개 병합)


,표기수,표기예
cmp_key,,
맘스터치,20,맘스터치 | 맘스터치(대송점) | 맘스터치(충장점) | 맘스터치(삼산점)
FONTERRA,18,FONTERRA LIMITED | Fonterra Limited(Fonterra Whareroa) |...
코스트코코리아,10,(주)코스트코코리아 | 코스트코코리아 | 코스트코 코리아 | (주)코스트코코리아(DELI)
다원,8,농업회사법인 다원주식회사 | 농업회사법인 주식회사 다원 | 농업회사법인다원주식회사 | 다원
남양주행복도시락,7,남양주 행복도시락 | 남양주행복도시락주식회사 | 남양주 행복 도시락 | 남양주행복도시락(주)
SCHREIBERFOODS,7,"SCHREIBER FOODS, INC. | SCHREIBER FOODS INC | SCHREIBER ..."
씨티푸드,7,주식회사 씨티푸드(제1공장) | (주)씨티푸드 | 주식회사 씨티푸드(제2공장) | (주) 씨티푸드
에스에프에스,6,농업회사법인 에스에프에스 주식회사(SFS) | 에스에프에스(SFS) 주식회사 | 에스에프에스(SFS)...
롯데푸드파스퇴르공장,6,롯데푸드(주)파스퇴르공장 | 롯데푸드(주) 파스퇴르공장 | 롯데푸드(주) 파스퇴르 공장 | 롯데푸드 ...


## 10-3. 업체별 행수·검출·검출률 — 검출 많은 순 (정규화 키 기준, 업체명 있는 행만)

In [4]:
def cmp_tbl(x):
    t = x.groupby("cmp_key").agg(행수=("pos", "size"), 검출=("pos", "sum"), 버섯행=("is_mush", "sum"),
                                 대표표기=("cmp_raw", lambda s: s.value_counts().index[0]))
    t["검출률(%)"] = (t["검출"] / t["행수"] * 100).round(3)
    t["검출비중(%)"] = (t["검출"] / t["검출"].sum() * 100).round(2)
    return t[["대표표기", "행수", "버섯행", "검출", "검출률(%)", "검출비중(%)"]]


has = df[df["cmp_key"].notna()]
cmp = cmp_tbl(has).sort_values(["검출", "행수"], ascending=False)
print(f"업체 {len(cmp):,}개 / 검출 1건 이상 업체 {(cmp['검출'] > 0).sum():,}개 / 검출비중 분모 {int(cmp['검출'].sum()):,}건")
s = cmp["검출"].cumsum() / cmp["검출"].sum() * 100
print(f"검출 상위 5개 업체 {s.iloc[4]:.1f}% / 10개 {s.iloc[9]:.1f}% / 50개 {s.iloc[49]:.1f}%")
display(cmp[cmp["검출"] > 0].head(50))

업체 16,847개 / 검출 1건 이상 업체 231개 / 검출비중 분모 535건
검출 상위 5개 업체 31.8% / 10개 39.3% / 50개 62.8%


,대표표기,행수,버섯행,검출,검출률(%),검출비중(%)
cmp_key,,,,,,
머쉬앤파머스,농업회사법인(주)머쉬앤파머스,1557,1106,73,4.689,13.64
이노팜,농업회사법인 이노팜 주식회사,446,110,40,8.969,7.48
살롱드파파푸드그룹,(주)살롱드파파푸드그룹,50,0,31,62.000,5.79
명덕농산,명덕농산,1214,743,14,1.153,2.62
이노그린,이노그린영농조합법인,78,67,12,15.385,2.24
TOYOREIZOFOODLOGISTICSHEADFACTORYTHEFOURTHREFRIGERATOR,"TOYOREIZO FOOD & LOGISTICS CO., LTD. HEAD FACTORY THE FO...",172,0,9,5.233,1.68
수월한,수월한 주식회사,26,0,9,34.615,1.68
함평천지버섯영농조합,함평천지버섯영농조합,185,185,8,4.324,1.50
사회복지법인평화의마을,사회복지법인평화의 마을,238,12,7,2.941,1.31


## 10-4. 업체 검출률 분포 — 행수 기준별 (전체 / 버섯 제외)

In [5]:
cmp_nm = cmp_tbl(has[~has["is_mush"]])
rows = []
for name, t in [("전체", cmp), ("버섯제외", cmp_nm)]:
    for th in [1, 30, 100, 1000]:
        x = t.loc[t["행수"] >= th, "검출률(%)"]
        rows.append({"구분": name, "최소행수": th, "업체수": len(x), "검출0건": int((x == 0).sum()),
                     "검출0건비율(%)": round((x == 0).mean() * 100, 1), "중앙값(%)": round(x.median(), 3),
                     "Q3(%)": round(x.quantile(.75), 3), "90분위(%)": round(x.quantile(.9), 3), "최대(%)": round(x.max(), 3)})
display(pd.DataFrame(rows))

,구분,최소행수,업체수,검출0건,검출0건비율(%),중앙값(%),Q3(%),90분위(%),최대(%)
0,전체,1,16847,16616,98.6,0.0,0.0,0.000,100.000
1,전체,30,2037,1934,94.9,0.0,0.0,0.000,62.000
2,전체,100,692,639,92.3,0.0,0.0,0.000,8.969
3,전체,1000,23,19,82.6,0.0,0.0,0.185,4.689
4,버섯제외,1,16811,16592,98.7,0.0,0.0,0.000,100.000
5,버섯제외,30,2028,1933,95.3,0.0,0.0,0.000,62.000
6,버섯제외,100,685,638,93.1,0.0,0.0,0.000,6.548
7,버섯제외,1000,20,19,95.0,0.0,0.0,0.000,0.147


## 10-5. 버섯 제외 — 검출률 높은 업체 (행수 ≥ 100) 와 검출 많은 업체

In [6]:
print("[검출률 상위, 행수 ≥ 100]")
display(cmp_nm[cmp_nm["행수"] >= 100].sort_values(["검출률(%)", "검출"], ascending=False).head(30))
print("[검출 건수 상위]")
display(cmp_nm.sort_values(["검출", "행수"], ascending=False).head(30))

[검출률 상위, 행수 ≥ 100]


,대표표기,행수,버섯행,검출,검출률(%),검출비중(%)
cmp_key,,,,,,
이노팜,농업회사법인 이노팜 주식회사,336,0,22,6.548,5.47
TOYOREIZOFOODLOGISTICSHEADFACTORYTHEFOURTHREFRIGERATOR,"TOYOREIZO FOOD & LOGISTICS CO., LTD. HEAD FACTORY THE FO...",172,0,9,5.233,2.24
사회복지법인평화의마을,사회복지법인평화의 마을,226,0,7,3.097,1.74
이싼푸드,이싼푸드,141,0,3,2.128,0.75
그린키친,(주)그린키친,104,0,2,1.923,0.50
꼬레햄,주식회사 꼬레햄,117,0,2,1.709,0.50
휴먼메쯔거라이,(주)휴먼메쯔거라이,240,0,4,1.667,1.00
우리식품,(주)우리식품,129,0,2,1.550,0.50
명덕농산,명덕농산,471,0,7,1.486,1.74


[검출 건수 상위]


,대표표기,행수,버섯행,검출,검출률(%),검출비중(%)
cmp_key,,,,,,
살롱드파파푸드그룹,(주)살롱드파파푸드그룹,50,0,31,62.000,7.71
이노팜,농업회사법인 이노팜 주식회사,336,0,22,6.548,5.47
TOYOREIZOFOODLOGISTICSHEADFACTORYTHEFOURTHREFRIGERATOR,"TOYOREIZO FOOD & LOGISTICS CO., LTD. HEAD FACTORY THE FO...",172,0,9,5.233,2.24
수월한,수월한 주식회사,26,0,9,34.615,2.24
명덕농산,명덕농산,471,0,7,1.486,1.74
사회복지법인평화의마을,사회복지법인평화의 마을,226,0,7,3.097,1.74
TOYOREIZOFOODLOGISTICSTOBISHIMAFACTORYTHE3RDREFRIGERATOR,"TOYOREIZO FOOD & LOGISTICS CO.,LTD. TOBISHIMAFACTORY THE...",99,0,7,7.071,1.74
맛잽이식품,주식회사 맛잽이식품,55,0,6,10.909,1.49
달구벌명가식품,주식회사 달구벌명가식품,26,0,6,23.077,1.49


## 10-6. 버섯 제외 검출 상위 10개 업체의 검출 품목(`train_prdlst_nm`)

In [7]:
top10 = cmp_nm.sort_values("검출", ascending=False).head(10).index
p = has[(has["pos"] == 1) & ~has["is_mush"] & has["cmp_key"].isin(top10)]
ct = pd.crosstab(p["cmp_key"].map(cmp_nm["대표표기"]), p["train_prdlst_nm"].astype(str))
display(ct.loc[ct.sum(axis=1).sort_values(ascending=False).index, ct.sum().sort_values(ascending=False).index])

train_prdlst_nm,식육가공품 및 포장육,기타기준규격외,"정액,수정란,종란",수산물
cmp_key,,,,
(주)살롱드파파푸드그룹,0,31,0,0
농업회사법인 이노팜 주식회사,0,0,22,0
"TOYOREIZO FOOD & LOGISTICS CO., LTD. HEAD FACTORY THE FOURTH REFRIGERATOR",0,0,0,9
수월한 주식회사,9,0,0,0
"TOYOREIZO FOOD & LOGISTICS CO.,LTD. TOBISHIMAFACTORY THE 3RD REFRIGERATOR",0,0,0,7
명덕농산,0,0,7,0
사회복지법인평화의 마을,7,0,0,0
농업회사법인주식회사천하제일한우,6,0,0,0
주식회사 달구벌명가식품,6,0,0,0


## 10-7. 눈에 띄는 업체 2곳의 검출 행 상세 — 이노팜, 살롱드파파푸드그룹

- 이노팜: 버섯 키워드(`is_mush`)로 걸러도 검출이 남는다 → 남는 행의 제품명이 무엇인지
- 살롱드파파푸드그룹: 50행 중 31건 검출 → 어떤 제품·검체이고 언제 몰렸는지

In [8]:
cols = ["is_mush", "train_prdlst_nm", "prduct_nm", "mnfctur_de", "addr"]
for key in ["이노팜", "살롱드파파푸드그룹"]:
    g = df[df["cmp_key"] == key]
    gp = g[g["pos"] == 1]
    print(f"===== {key} =====  전체 {len(g):,}행 / 검출 {len(gp)}건 (버섯 키워드 {int(gp['is_mush'].sum())}건 / 키워드 없음 {int((~gp['is_mush']).sum())}건)")
    print("[주소]")
    display(gp["addr"].astype(str).value_counts().rename("검출건수").to_frame())
    print("[연월별 검출]")
    display(gp["mnfctur_de"].dt.to_period("M").value_counts().sort_index().rename("검출건수").to_frame())
    print("[버섯 키워드 없는 검출 행 — 제품명]")
    display(gp.loc[~gp["is_mush"], cols].sort_values("mnfctur_de").reset_index(drop=True))
    if gp["is_mush"].any():
        print("[버섯 키워드 걸린 검출 행 — 제품명]")
        display(gp.loc[gp["is_mush"], cols].sort_values("mnfctur_de").reset_index(drop=True))

===== 이노팜 =====  전체 446행 / 검출 40건 (버섯 키워드 18건 / 키워드 없음 22건)
[주소]


,검출건수
addr,
전라남도 곡성군 석곡면 노치로 277,40


[연월별 검출]


,검출건수
mnfctur_de,
2024-10,2
2024-11,2
2025-04,1
2025-05,8
2025-07,2
2025-09,9
2025-10,13
2026-01,3


[버섯 키워드 없는 검출 행 — 제품명]


,is_mush,train_prdlst_nm,prduct_nm,mnfctur_de,addr
0,False,"정액,수정란,종란",생육실2-3,2024-10-25,전라남도 곡성군 석곡면 노치로 277
1,False,"정액,수정란,종란",생육실2-5,2024-10-25,전라남도 곡성군 석곡면 노치로 277
2,False,"정액,수정란,종란",균긁기실 배수구 (Scratch Room Drain) 1번,2024-11-22,전라남도 곡성군 석곡면 노치로 277
3,False,"정액,수정란,종란",균긁기 칼날 2차라인 (2nd Line of Scraper Blades) 2번,2024-11-22,전라남도 곡성군 석곡면 노치로 277
4,False,"정액,수정란,종란",균긁기실바닥 1,2025-07-11,전라남도 곡성군 석곡면 노치로 277
5,False,"정액,수정란,종란",균긁기실바닥 2,2025-07-11,전라남도 곡성군 석곡면 노치로 277
6,False,"정액,수정란,종란",컨베이어1,2025-09-08,전라남도 곡성군 석곡면 노치로 277
7,False,"정액,수정란,종란",컨베이어2,2025-09-08,전라남도 곡성군 석곡면 노치로 277
8,False,"정액,수정란,종란",포장기2,2025-09-08,전라남도 곡성군 석곡면 노치로 277
9,False,"정액,수정란,종란",생육실2,2025-09-08,전라남도 곡성군 석곡면 노치로 277


... (4 outputs left out)

[연월별 검출]


,검출건수
mnfctur_de,
2025-07,10
2025-08,17
2025-09,4


[버섯 키워드 없는 검출 행 — 제품명]


,is_mush,train_prdlst_nm,prduct_nm,mnfctur_de,addr
0,False,기타기준규격외,간장연어장 1번,2025-07-21,"광주광역시 광산구 임방울대로358번길 28 , 1층"
1,False,기타기준규격외,간장연어장 2번,2025-07-21,"광주광역시 광산구 임방울대로358번길 28 , 1층"
2,False,기타기준규격외,간장연어장 1번,2025-07-23,"광주광역시 광산구 임방울대로358번길 28 , 1층"
3,False,기타기준규격외,간장연어장,2025-07-25,"광주광역시 광산구 임방울대로358번길 28 , 1층"
4,False,기타기준규격외,간장연어장 2,2025-07-28,"광주광역시 광산구 임방울대로358번길 28 , 1층"
5,False,기타기준규격외,간장연어장 1,2025-07-28,"광주광역시 광산구 임방울대로358번길 28 , 1층"
6,False,기타기준규격외,연어포장,2025-07-30,"광주광역시 광산구 임방울대로358번길 28 , 1층"
7,False,기타기준규격외,이산화염소100ppm15분연어,2025-07-30,"광주광역시 광산구 임방울대로358번길 28 , 1층"
8,False,기타기준규격외,연어장-세척(야채없음),2025-07-31,"광주광역시 광산구 임방울대로358번길 28 , 1층"
9,False,기타기준규격외,연어장-휘발(야채없음),2025-07-31,"광주광역시 광산구 임방울대로358번길 28 , 1층"


## 정리 (셀 출력 + 별도 조회로 확인한 수치만)

- **업체명 결측**: 332,410행(49.1%)인데 검출은 1,434건으로 전체 검출의 **72.8%**. 검출률도 결측 0.431% vs 있음 0.155% → 결측이 무작위가 아님. 업체명 기반 분석은 검출의 27%(535건)만 보는 셈
- 정규화: 고유 업체명 19,225 → 16,847개 (2,378개 병합). 맘스터치(지점 20개 표기) 같은 체인은 지점이 하나로 합쳐짐
- 업체명 있는 16,847개 중 검출 1건 이상 231개. 상위 5개 업체가 535건 중 31.8%, 50개가 62.8%
- 행수 ≥ 1,000 업체 23개 중 19개가 검출 0건
- 눈에 띄는 업체 (prduct_nm 별도 조회로 확인)
  - 머쉬앤파머스: 1,557행 중 버섯 1,106행, 검출 73건(4.689%) — 청도 팽이버섯 업체
  - 이노팜(곡성군): 키워드상 버섯행 0인 검출 22건이 남지만, 제품명이 `생육실 2-1`, `컨베이어1`, `포장실 20` 등 → 버섯 농가 **환경 검체**. 버섯 키워드 기준이 이런 행을 못 잡음
  - 살롱드파파푸드그룹(광주 광산구): 50행 중 31건 검출(62%), 2025-07~09 에 몰림. 제품명 `간장연어장`, `연어포장`, `클린콜`, `이산화세척` 등 → 연어장 제품 + 세척·소독 공정 검체. 노트북 9 의 광산구 버섯제외 고검출률(0.931%)과 연결됨
